# xgov: full functional dump

Loads the real `xgov` approval program (from `tests/contracts/xgov/`), builds an isolated immutable presentation view, then prints its functional representation.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent.parent / "src"))

import tealql.tealtools.ssa as teal_ssa
from tealql.tealtools.analysis import DerivedProfile, derived_program

In [2]:
DB = HERE.parent.parent / "tests" / "contracts" / "xgov"

canonical = teal_ssa.SSAProgram(DB)
p = derived_program(canonical, DerivedProfile.PRESENTATION)

print(f"{len(p)} assignments, {len(p.vars)} SSA vars, {len(p.phis)} phis, {len(p.blocks)} BBs")

1061 assignments, 794 SSA vars, 10 phis, 167 BBs


In [3]:
print(p.functional())

L   2: intcblock 0 1 10 3 ()
L   3: bytecblock 0x766f74655f74797065 0x 0x6f75616964 0x766f74655f6964 0x6f7074696f6e5f636f756e7473 0x69735f626f6f747374726170706564 0x766f7465725f636f756e74 0x636c6f73655f74696d65 0x746f74616c5f6f7074696f6e73 0x56 0x736e617073686f745f7075626c69635f6b6579 0x6d657461646174615f697066735f636964 0x73746172745f74696d65 0x656e645f74696d65 0x71756f72756d 0x6e66745f696d6167655f75726c 0x4c6bea72 0x151f7c75 0x6e66745f61737365745f6964 0x068101 0x2c ()
L   4: V#1@L4 = txn NumAppArgs ()
L   5: V#1@L5 = intc_0 ()
L   6: V#1@L6 = == (0, V#1@L4)
L   7: bnz main_l14 (V#1@L6)
L   8: V#1@L8 = txna ApplicationArgs 0 ()
L   9: V#1@L9 = pushbytes 0x101cea00 ()
L  10: V#1@L10 = == (0x101cea00, V#1@L8)
L  11: bnz main_l13 (V#1@L10)
L  12: V#1@L12 = txna ApplicationArgs 0 ()
L  13: V#1@L13 = pushbytes 0x5d4cf066 ()
L  14: V#1@L14 = == (0x5d4cf066, V#1@L12)
L  15: bnz main_l12 (V#1@L14)
L  16: V#1@L16 = txna ApplicationArgs 0 ()
L  17: V#1@L17 = pushbytes 0xa4e8d164 ()
L  18: V#1@L

## Range analysis

`propagate_ranges()` is an independent pass that tags SSA vars with a static integer range and type. For now it seeds only from boolean-returning ops (`<`, `>`, `==`, `&&`, …) — their single output gets `range = [0..1]`, `type = uint64` — and unions through phis to fixed point. Rendering with `show_ranges=True` attaches a `/*[var<=1]*/` annotation next to each annotated var.


In [4]:
p.propagate_ranges()

n_var = sum(1 for v in p.vars.values() if v.range is not None)
n_phi = sum(1 for ph in p.phis.values() if ph.range is not None)
print(f"{n_var}/{len(p.vars)} SSA vars have a range, {n_phi}/{len(p.phis)} phis have a range")


144/794 SSA vars have a range, 0/10 phis have a range


Functional dump for the dispatcher prologue (lines 1–30) with range annotations turned on. Note the comparison results (`==` outputs) decorated with `/*[V<=1]*/`, and how the same annotation follows them into the `bnz` consumers.


In [5]:
print(p.functional(show_ranges=True))


L   2: intcblock 0 1 10 3 ()
L   3: bytecblock 0x766f74655f74797065 0x 0x6f75616964 0x766f74655f6964 0x6f7074696f6e5f636f756e7473 0x69735f626f6f747374726170706564 0x766f7465725f636f756e74 0x636c6f73655f74696d65 0x746f74616c5f6f7074696f6e73 0x56 0x736e617073686f745f7075626c69635f6b6579 0x6d657461646174615f697066735f636964 0x73746172745f74696d65 0x656e645f74696d65 0x71756f72756d 0x6e66745f696d6167655f75726c 0x4c6bea72 0x151f7c75 0x6e66745f61737365745f6964 0x068101 0x2c ()
L   4: V#1@L4 /*[V#1@L4<=16]*/ = txn NumAppArgs ()
L   5: V#1@L5 = intc_0 ()
L   6: V#1@L6 /*[V#1@L6<=1]*/ = == (0, V#1@L4 /*[V#1@L4<=16]*/)
L   7: bnz main_l14 (V#1@L6 /*[V#1@L6<=1]*/)
L   8: V#1@L8 = txna ApplicationArgs 0 ()
L   9: V#1@L9 = pushbytes 0x101cea00 ()
L  10: V#1@L10 /*[V#1@L10<=1]*/ = == (0x101cea00, V#1@L8)
L  11: bnz main_l13 (V#1@L10 /*[V#1@L10<=1]*/)
L  12: V#1@L12 = txna ApplicationArgs 0 ()
L  13: V#1@L13 = pushbytes 0x5d4cf066 ()
L  14: V#1@L14 /*[V#1@L14<=1]*/ = == (0x5d4cf066, V#1@L12)
L  15: bn